# Econ 524 — Python Crash Course
### NumPy · pandas (incl. sparse & groupby) · scipy.stats / statsmodels

**Author: Joel Reyes

**Who this is for.** As a student who already know Python syntax — loops, functions, list comprehensions.
This notebook is not about syntax. It's about the handful of NumPy/pandas/stats idioms that show
up in *every* empirical script you'll write this semester, plus a few habits (vectorize, don't
loop; know your data's density; check `p/n` before you regress) that will save you real time.

**Data.** We'll use one real dataset throughout — the March 2015 CPS wage sample from
Lab 1 ($n=5150$) — so the examples build on each other instead of being disconnected snippets.



In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import matplotlib.pyplot as plt

rng = np.random.default_rng(524)


In [2]:
URL = ("https://raw.githubusercontent.com/CausalAIBook/MetricsMLNotebooks/"
       "main/data/wage2015_subsample_inference.csv")
wage = pd.read_csv(URL)
print(wage.shape)
wage.head(3)

(5150, 20)


,wage,lwage,sex,shs,hsg,scl,clg,ad,mw,so,we,ne,exp1,exp2,exp3,exp4,occ,occ2,ind,ind2
0,9.615385,2.263364,1,0,0,0,1,0,0,0,0,1,7.0,0.49,0.343,0.2401,3600.0,11,8370.0,18
1,48.076923,3.872802,0,0,0,0,1,0,0,0,0,1,31.0,9.61,29.791,92.3521,3050.0,10,5070.0,9
2,11.057692,2.403126,0,0,1,0,0,0,0,0,0,1,18.0,3.24,5.832,10.4976,6260.0,19,770.0,4


---
# Part 1 — NumPy: arrays, vectorization, linear algebra

The mental model for the whole course: **data is a matrix, operations are matrix operations.**
Avoid `for` loops over rows whenever there's a vectorized equivalent — it's not just faster, it's
also closer to how you'll write down the math.

## 1.1 Arrays, shape, axes

`axis=0` collapses rows (→ one value per column). `axis=1` collapses columns (→ one value per
row). This trips almost everyone up at least once.

In [3]:
A = np.arange(12).reshape(3, 4)
print(A)
print("column sums (axis=0):", A.sum(axis=0))
print("row sums    (axis=1):", A.sum(axis=1))
print("overall mean / sd  :", A.mean()," ",  A.std())

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
column sums (axis=0): [12 15 18 21]
row sums    (axis=1): [ 6 22 38]
overall mean / sd  : 5.5   3.452052529534663


## 1.2 Vectorization vs. loops

Same computation, two ways. 

In [4]:
n = 200_000
x = rng.standard_normal(n)

%timeit -n 3 -r 3 sum(xi**2 for xi in x)      # pure python loop
%timeit -n 3 -r 3 (x**2).sum()                 # vectorized

14.9 ms ± 1.04 ms per loop (mean ± std. dev. of 3 runs, 3 loops each)
98.6 μs ± 14.7 μs per loop (mean ± std. dev. of 3 runs, 3 loops each)


**Rule of thumb:** if you're writing `for i in range(len(array))`, stop and ask whether the
operation has a NumPy equivalent. It almost always does.

## 1.3 Broadcasting

NumPy lets arrays of *different but compatible* shapes interact without an explicit loop. The
rule: shapes are compared from the right; a dimension of size 1 (or missing) stretches to match.



In [5]:
Z = rng.standard_normal((5, 3)) * np.array([10, 1, 0.1]) + np.array([100, 0, -5])
print("raw:\n", Z)

col_mean = Z.mean(axis=0)          # shape (3,)
col_std  = Z.std(axis=0)           # shape (3,)
Z_std = (Z - col_mean) / col_std   # (5,3) - (3,) broadcasts to (5,3)

print("\nstandardized column means (~0):", Z_std.mean(axis=0).round(10))
print("standardized column sds   (~1):", Z_std.std(axis=0).round(10))

raw:
 [[ 9.04963478e+01  1.69845205e-01 -5.05199716e+00]
 [ 9.23686088e+01  1.27113230e+00 -4.81052562e+00]
 [ 1.14698843e+02 -1.31978544e+00 -5.10940539e+00]
 [ 1.10343490e+02  9.96788032e-01 -5.01201399e+00]
 [ 1.09783551e+02  9.08417562e-02 -4.98422294e+00]]

standardized column means (~0): [ 0. -0. -0.]
standardized column sds   (~1): [1. 1. 1.]


## 1.4 Boolean and fancy indexing

No `if` statements needed — build a boolean array and index with it.

In [6]:
v = np.arange(20)
mask = (v % 3 == 0)
print("mask       :", mask)
print("v[mask]    :", v[mask])
print("v[v > 15]  :", v[v > 15])

# combine conditions with & and | (not `and`/`or` — those don't vectorize)
print("v[(v>5)&(v<12)]:", v[(v > 5) & (v < 12)])

mask       : [ True False False  True False False  True False False  True False False
  True False False  True False False  True False]
v[mask]    : [ 0  3  6  9 12 15 18]
v[v > 15]  : [16 17 18 19]
v[(v>5)&(v<12)]: [ 6  7  8  9 10 11]


## 1.5 Linear algebra — this is where OLS lives

Three equivalent ways to compute $\hat\beta = (X'X)^{-1}X'y$. 

In [7]:
X = rng.standard_normal((100, 3))
beta_true = np.array([1.0, -2.0, 0.5])
y = X @ beta_true + 0.5 * rng.standard_normal(100)

b1 = np.linalg.inv(X.T @ X) @ (X.T @ y)     # textbook formula — avoid in practice
b2 = np.linalg.solve(X.T @ X, X.T @ y)      # better: solves the linear system directly
b3 = np.linalg.lstsq(X, y, rcond=None)[0]   # best: works even if X'X is singular

print(b1)
print(b2)
print(b3)

[ 1.00819846 -2.02371885  0.48109056]
[ 1.00819846 -2.02371885  0.48109056]
[ 1.00819846 -2.02371885  0.48109056]


> **Gotcha.** `np.linalg.inv(X.T @ X)` will silently return garbage (or raise
> `LinAlgError: Singular matrix`) once $p \ge n$ or columns are collinear.

## 1.6 Random number generation


In [8]:
rng1 = np.random.default_rng(1)
rng2 = np.random.default_rng(1)
print(np.allclose(rng1.standard_normal(5), rng2.standard_normal(5)))   # same seed -> same draws

# common distributions you'll use all semester
print("normal :", rng.normal(loc=0, scale=1, size=3))
print("uniform:", rng.uniform(0, 1, size=3))
print("binom  :", rng.binomial(n=1, p=0.3, size=5))
print("choice :", rng.choice(['A', 'B', 'C'], size=5, p=[0.2, 0.3, 0.5]))

True
normal : [ 0.33701946 -0.01928836  0.56065725]
uniform: [0.4465007  0.26467884 0.97471172]
binom  : [1 0 1 0 0]
choice : ['C' 'C' 'C' 'C' 'B']


---
# Part 2 — pandas: handling, reshaping, grouping data

We'll work with the CPS wage data loaded above. Columns worth knowing: `wage`, `lwage` (log
wage), `sex`, education dummies (`shs, hsg, scl, clg, ad`), `exp1..exp4` (experience and powers),
`occ2`/`ind2` (occupation/industry codes), region dummies (`mw, so, we, ne`).

## 2.1 Selection: `.loc` vs `.iloc` vs boolean masks

- `.loc[row_label, col_label]` — label-based.
- `.iloc[row_pos, col_pos]` — position-based, like NumPy.
- boolean mask — the one you'll use constantly for filtering.

In [9]:
print(wage.loc[0:2, ['wage', 'sex']])          # label-based; note: 0:2 is INCLUSIVE with .loc
print()
print(wage.iloc[0:2, 0:2])                       # position-based; 0:2 EXCLUSIVE, like numpy

# boolean filter: college-educated women
sub = wage[(wage['sex'] == 1) & (wage['clg'] == 1)]
print(f"\n{len(sub)} rows match")
sub[['wage', 'exp1']].describe()

        wage  sex
0   9.615385    1
1  48.076923    0
2  11.057692    0

        wage     lwage
0   9.615385  2.263364
1  48.076923  3.872802

795 rows match


,wage,exp1
count,795.000000,795.000000
mean,25.226671,11.480503
std,14.655870,10.439368
min,4.807692,1.000000
25%,16.490746,3.000000
50%,21.634615,8.000000
75%,29.385198,17.500000
max,111.538462,40.000000


> **Gotcha.** `.loc` slicing is *inclusive* of the endpoint; `.iloc` and NumPy slicing are not.
> `df.loc[0:2]` gives you 3 rows; `df.iloc[0:2]` gives you 2. This bites everyone at least once.

## 2.2 Vectorized `.apply` / `assign` — and when to avoid `.apply`

`.apply` runs a Python function row-by-row — it's convenient but it's a loop in disguise. Prefer a
vectorized expression when one exists.

In [10]:
# slow-ish: apply with a python function
%timeit -n 3 -r 3 wage['wage'].apply(lambda w: np.log(w) if w > 0 else np.nan)

# fast: vectorized numpy ufunc
%timeit -n 3 -r 3 np.log(wage['wage'].where(wage['wage'] > 0))

# .assign() for building a pipeline of new columns without mutating in place
wage2 = wage.assign(
    lwage_check = lambda d: np.log(d['wage']),
    high_exp    = lambda d: (d['exp1'] > 15).astype(int),
)
wage2[['wage', 'lwage', 'lwage_check', 'high_exp']].head(3)

859 μs ± 35.6 μs per loop (mean ± std. dev. of 3 runs, 3 loops each)
249 μs ± 17.7 μs per loop (mean ± std. dev. of 3 runs, 3 loops each)


,wage,lwage,lwage_check,high_exp
0,9.615385,2.263364,2.263364,0
1,48.076923,3.872802,3.872802,1
2,11.057692,2.403126,2.403126,1


## 2.3 Missing data

Real data has gaps. The core toolkit: `.isna()`, `.dropna()`, `.fillna()`.

In [11]:
d = wage.copy()
missing_idx = d.sample(50, random_state=1).index
d.loc[missing_idx, 'lwage'] = np.nan

print("missing count:", d['lwage'].isna().sum())
print("rows if dropped:", len(d.dropna(subset=['lwage'])))

d['lwage_filled'] = d['lwage'].fillna(d['lwage'].median())     # simple imputation
print("still missing after fillna:", d['lwage_filled'].isna().sum())

# in a regression context you'd more often just drop:
d_complete = d.dropna(subset=['lwage'])

missing count: 50
rows if dropped: 5100
still missing after fillna: 0


> **Gotcha.** `fillna` with the mean/median silently changes your variance — fine for a quick
> plot, usually *not* fine to hand straight to a regression without at least flagging it (e.g. an
> `is_imputed` dummy). We'll come back to principled missing-data handling later in the course.

## 2.4 GroupBy: split–apply–combine

This is the single most useful pandas idiom for applied work. Three flavors:

1. **`.agg`** — one summary number per group.
2. **`.transform`** — a value *per row*, computed within its group (same length as input — great
   for within-group standardization).
3. **pivot tables** — `.agg` reshaped into a 2-D grid.

In [12]:
# 1. agg: several summaries at once, named columns
summary = wage.groupby('sex').agg(
    mean_wage = ('wage', 'mean'),
    median_wage = ('wage', 'median'),
    sd_wage = ('wage', 'std'),
    n = ('wage', 'size'),
)
summary

,mean_wage,median_wage,sd_wage,n
sex,,,,
0,24.019261,19.230769,23.148700,2861
1,22.649413,18.846154,17.940375,2289


In [13]:
# 2. transform: e.g. z-score wages WITHIN each sex group (same length as `wage`)
wage['wage_z_within_sex'] = wage.groupby('sex')['wage'].transform(lambda x: (x - x.mean()) / x.std())
wage[['sex', 'wage', 'wage_z_within_sex']].head()

,sex,wage,wage_z_within_sex
0,1,9.615385,-0.726519
1,0,48.076923,1.039266
2,0,11.057692,-0.559926
3,1,13.942308,-0.485336
4,1,28.846154,0.345408


In [14]:
# multiple grouping keys + multiple summary columns
g2 = wage.groupby(['sex', 'mw']).agg(mean_lwage=('lwage', 'mean'), n=('lwage', 'size'))
g2

mean_lwage     n
sex mw                  
0   0     2.999133  2120
    1     2.955491   741
1   0     2.979812  1693
    1     2.863338   596

In [15]:
# 3. pivot_table: same idea, reshaped to a grid — great for quick cross-tabs
pt = wage.pivot_table(index='sex', columns='mw', values='lwage', aggfunc='mean')
pt.rename(index={0: 'Male', 1: 'Female'}, columns={0: 'Not Midwest', 1: 'Midwest'})

mw,Not Midwest,Midwest
sex,,
Male,2.999133,2.955491
Female,2.979812,2.863338


> **Gotcha.** `.agg('mean')` on a groupby drops non-numeric columns automatically in modern
> pandas, but relying on that is fragile — be explicit about which columns you want, as above with
> the `col=('source_col','func')` syntax.

## 2.5 Merging and concatenating

`merge` joins on a key (like a SQL join); `concat` stacks frames. You'll use `merge` constantly
when combining, e.g., an outcomes dataset with a covariates dataset that share an ID.

In [16]:
left  = pd.DataFrame({'id': [1, 2, 3], 'x': [10, 20, 30]})
right = pd.DataFrame({'id': [2, 3, 4], 'y': [200, 300, 400]})

print("inner (only matching ids):\n", pd.merge(left, right, on='id', how='inner'))
print("\nleft (keep all of `left`, fill NaN where no match):\n", pd.merge(left, right, on='id', how='left'))
print("\nconcat (stack rows, union of columns):\n", pd.concat([left, right], axis=0, ignore_index=True))

inner (only matching ids):
    id   x    y
0   2  20  200
1   3  30  300

left (keep all of `left`, fill NaN where no match):
    id   x      y
0   1  10    NaN
1   2  20  200.0
2   3  30  300.0

concat (stack rows, union of columns):
    id     x      y
0   1  10.0    NaN
1   2  20.0    NaN
2   3  30.0    NaN
3   2   NaN  200.0
4   3   NaN  300.0
5   4   NaN  400.0


> **Gotcha.** After a `merge`, always sanity-check `len(result)`. A key that isn't unique on one
> side silently fans out into a many-to-many join and multiplies your row count — a very common,
> very quiet bug.

## 2.6 Categorical data and dummy encoding

`occ2`/`ind2` are occupation/industry *codes* — integers, but not meaningfully ordered. Two tools:
`.astype('category')` (memory-efficient, model-aware) and `pd.get_dummies` (one-hot encoding,
what feeds directly into the design matrices from Lab 1).

In [17]:
wage['occ2'] = wage['occ2'].astype('category')
print(wage['occ2'].dtype, "-", wage['occ2'].nunique(), "categories")

dummies = pd.get_dummies(wage['occ2'], prefix='occ')
print(dummies.shape)
dummies.iloc[:3, :5]

category - 22 categories
(5150, 22)


,occ_1,occ_2,occ_3,occ_4,occ_5
0,False,False,False,False,False
1,False,False,False,False,False
2,False,False,False,False,False


## 2.7 Sparse data structures

One-hot-encoding a categorical variable with many levels (22 occupation codes × 5150 rows) produces a matrix that is
almost entirely zeros. Storing it densely wastes memory for no reason — pandas' sparse dtype fixes
that.

In [18]:
dense_dummies  = pd.get_dummies(wage['occ2'], prefix='occ', sparse=False)
sparse_dummies = pd.get_dummies(wage['occ2'], prefix='occ', sparse=True)

mem_dense  = dense_dummies.memory_usage(deep=True).sum() / 1024
mem_sparse = sparse_dummies.memory_usage(deep=True).sum() / 1024
print(f"dense  : {mem_dense:6.1f} KB")
print(f"sparse : {mem_sparse:6.1f} KB   ({mem_sparse/mem_dense:.0%} of dense)")
print("dtype of one sparse column:", sparse_dummies.dtypes.iloc[0])

dense  :  110.8 KB
sparse :   25.3 KB   (23% of dense)
dtype of one sparse column: Sparse[bool, False]


In [19]:
# a SparseArray directly, with an explicit fill_value (default fill is NaN, not 0 — watch this)
raw = np.zeros(20)
raw[[2, 5, 9]] = [1.5, 2.3, 4.1]

sp = pd.arrays.SparseArray(raw, fill_value=0.0)
print(sp)
print("density (fraction non-fill):", sp.density)

[0.0, 0.0, 1.5, 0.0, 0.0, 2.3, 0.0, 0.0, 0.0, 4.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Fill: 0.0
IntIndex
Indices: array([2, 5, 9], dtype=int32)

density (fraction non-fill): 0.15


> **Gotcha.** `pd.arrays.SparseArray(x)` defaults to `fill_value=np.nan`, not `0.0`. If your
> array's "empty" value is zero (the usual case for dummy encodings), pass `fill_value=0.0`
> explicitly, or `density` will silently report 1.0 and you'll wonder why sparsity "isn't working."
>


## 2.8 Reshaping: wide ↔ long

`melt` (wide→long) and `pivot` (long→wide) are inverses. Useful whenever you're comparing several
model specifications' worth of output — exactly the shape you'd want to build Lab 1's summary
table for plotting.

In [20]:
results_wide = pd.DataFrame({
    'model': ['Basic', 'Flexible', 'Extra flex'],
    'R2_in': [0.345, 0.398, 0.542],
    'R2_out': [0.269, 0.187, -0.405],
})
long = results_wide.melt(id_vars='model', var_name='sample', value_name='R2')
print(long)

back_to_wide = long.pivot(index='model', columns='sample', values='R2')
back_to_wide

        model  sample     R2
0       Basic   R2_in  0.345
1    Flexible   R2_in  0.398
2  Extra flex   R2_in  0.542
3       Basic  R2_out  0.269
4    Flexible  R2_out  0.187
5  Extra flex  R2_out -0.405


sample,R2_in,R2_out
model,,
Basic,0.345,0.269
Extra flex,0.542,-0.405
Flexible,0.398,0.187


---
# Part 3 — scipy.stats and statsmodels

NumPy/pandas get your data into shape. `scipy.stats` gives you distributions and classical tests;
`statsmodels` gives you regression output that looks like what you'd see in Stata or R, with
standard errors, t-stats, and (crucially for this course) a choice of covariance estimator.

## 3.1 Distributions: `pdf`, `cdf`, `ppf`, `rvs`

Every `scipy.stats` distribution object supports the same four methods. `ppf` (percent-point
function) is the inverse CDF — what you use for critical values.

In [21]:
print("P(Z <= 1.96)         :", stats.norm.cdf(1.96).round(4))
print("97.5th percentile of Z :", stats.norm.ppf(0.975).round(4))   # the familiar 1.96
print("density at 0           :", stats.norm.pdf(0).round(4))

draws = stats.norm.rvs(loc=0, scale=1, size=1000, random_state=0)
print("simulated mean, sd     :", draws.mean().round(3), draws.std().round(3))

# t-distribution matters once you have a small sample or are estimating sigma
print("t critical value, df=10:", stats.t.ppf(0.975, df=10).round(3), " (vs normal:", round(stats.norm.ppf(0.975),3), ")")

P(Z <= 1.96)         : 0.975
97.5th percentile of Z : 1.96
density at 0           : 0.3989
simulated mean, sd     : -0.045 0.987
t critical value, df=10: 2.228  (vs normal: 1.96 )


## 3.2 Hypothesis tests you'll actually use

Two-sample t-test, correlation test, and a chi-squared test of independence — the classical
building blocks before you get to regression-based inference.

In [22]:
men   = wage.loc[wage['sex'] == 0, 'lwage']
women = wage.loc[wage['sex'] == 1, 'lwage']

t_stat, p_val = stats.ttest_ind(men, women)
print(f"t-test (mean lwage, men vs women): t = {t_stat:.3f}, p = {p_val:.4f}")

r, p_corr = stats.pearsonr(wage['exp1'], wage['lwage'])
print(f"corr(experience, lwage): r = {r:.3f}, p = {p_corr:.4f}")

table = pd.crosstab(wage['sex'], wage['clg'])
chi2, p_chi, dof, expected = stats.chi2_contingency(table)
print(f"\nchi-sq test (sex vs college): chi2 = {chi2:.2f}, p = {p_chi:.4f}")
print(table)

t-test (mean lwage, men vs women): t = 2.398, p = 0.0165
corr(experience, lwage): r = 0.070, p = 0.0000

chi-sq test (sex vs college): chi2 = 16.46, p = 0.0000
clg     0    1
sex           
0    2020  841
1    1494  795


## 3.3 statsmodels: OLS with a real regression table

This is your workhorse for the rest of the semester. `sm.add_constant` adds the intercept column
explicitly — statsmodels does **not** add one automatically, unlike, e.g., `statsmodels.formula.api`.

In [23]:
X = sm.add_constant(wage[['exp1', 'exp2', 'clg', 'sex']])
y = wage['lwage']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.052
Model:                            OLS   Adj. R-squared:                  0.051
Method:                 Least Squares   F-statistic:                     70.87
Date:                Fri, 28 Aug 2026   Prob (F-statistic):           1.63e-58
Time:                        09:15:14   Log-Likelihood:                -4277.5
No. Observations:                5150   AIC:                             8565.
Df Residuals:                    5145   BIC:                             8598.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.7987      0.022    126.814      0.0

## 3.4 Choosing a covariance estimator

`.fit()` defaults to classical (homoskedastic) standard errors. Pass `cov_type` to get any of the
heteroskedasticity-robust flavors.

In [24]:
for cov in ['nonrobust', 'HC0', 'HC1', 'HC2', 'HC3']:
    m = sm.OLS(y, X).fit(cov_type=cov)
    print(f"{cov:9s}  se(exp1) = {m.bse['exp1']:.5f}")

nonrobust  se(exp1) = 0.00271
HC0        se(exp1) = 0.00274
HC1        se(exp1) = 0.00274
HC2        se(exp1) = 0.00274
HC3        se(exp1) = 0.00274




Also available directly from a fitted model, without recomputing anything:

In [25]:
print("params      :\n", model.params.round(4))
print("\nconf. int.  :\n", model.conf_int().round(4))
print("\nR-squared   :", round(model.rsquared, 4))
print("fitted values (first 3):", model.fittedvalues.values[:3].round(3))
print("residuals (first 3)    :", model.resid.values[:3].round(3))

params      :
 const    2.7987
exp1     0.0110
exp2    -0.0141
clg      0.2703
sex     -0.0513
dtype: float64

conf. int.  :
             0       1
const  2.7555  2.8420
exp1   0.0057  0.0163
exp2  -0.0280 -0.0001
clg    0.2369  0.3038
sex   -0.0819 -0.0207

R-squared   : 0.0522
fitted values (first 3): [3.088 3.275 2.951]
residuals (first 3)    : [-0.825  0.598 -0.548]


## 3.5 `formula` API — R-style syntax, useful for building design matrices fast
`*` includes
main effects and the interaction; `**2` expands all pairwise interactions among a set of terms.

In [26]:
import statsmodels.formula.api as smf

m_formula = smf.ols('lwage ~ sex + exp1 + exp2 + C(occ2)', data=wage).fit()
print(m_formula.params.head())
print("\nnumber of parameters:", len(m_formula.params))

Intercept       3.172238
C(occ2)[T.2]    0.050071
C(occ2)[T.3]    0.090155
C(occ2)[T.4]    0.002788
C(occ2)[T.5]    0.026437
dtype: float64

number of parameters: 25


`C(occ2)` tells the formula API to treat `occ2` as categorical (dummy-encode it) even though
it's stored as integers — same idea as `.astype('category')` in §2.6, just inline.

---
# Part 4 — Putting it together: a mini pipeline

One end-to-end example touching every section above: build features (pandas), compute something
by hand to check statsmodels isn't a black box (NumPy), run inference under two covariance
assumptions (stats), and look at group heterogeneity (groupby).

In [27]:
# 1. Feature engineering with pandas
df = wage.copy()
df['occ2'] = df['occ2'].astype('category')
X_df = pd.get_dummies(df[['sex', 'exp1', 'exp2', 'clg', 'occ2']], columns=['occ2'],
                       drop_first=True, sparse=True)
X_df = sm.add_constant(X_df.astype(float))   # statsmodels needs a float dtype, not sparse-bool
y = df['lwage'].values

print("design matrix shape:", X_df.shape, "  p/n =", round((X_df.shape[1]-1)/len(df), 3))

design matrix shape: (5150, 26)   p/n = 0.005


In [28]:
# 2. Solve by hand with NumPy, confirm it matches statsmodels
Xmat = X_df.values.astype(float)
beta_numpy = np.linalg.lstsq(Xmat, y, rcond=None)[0]

model_full = sm.OLS(y, X_df).fit()
beta_sm = model_full.params.values

print("max |numpy - statsmodels| coefficient gap:", np.max(np.abs(beta_numpy - beta_sm)))

max |numpy - statsmodels| coefficient gap: 2.7755575615628914e-15


In [29]:
# 3. Robust vs. classical inference on the 'sex' coefficient
for cov in ['nonrobust', 'HC1']:
    m = sm.OLS(y, X_df).fit(cov_type=cov)
    ci = m.conf_int().loc['sex']
    print(f"{cov:9s}  beta_sex = {m.params['sex']:.4f},  95% CI = [{ci[0]:.4f}, {ci[1]:.4f}]")

nonrobust  beta_sex = -0.0658,  95% CI = [-0.0966, -0.0349]
HC1        beta_sex = -0.0658,  95% CI = [-0.0972, -0.0343]


In [30]:
# 4. groupby to see whether the gender gap in the RAW data varies by education
gap_by_educ = (df.groupby('clg')['lwage']
                 .apply(lambda s: s[df.loc[s.index, 'sex']==0].mean() - s[df.loc[s.index, 'sex']==1].mean()))
gap_by_educ.index = ['Not college grad', 'College grad']
gap_by_educ.rename('raw log-wage gap (men - women)').to_frame()

,raw log-wage gap (men - women)
Not college grad,0.045961
College grad,0.062361


**Note the honest caveat**, which is itself a preview of the rest of the course: the raw gap in
the last cell is *not* a causal estimate of anything — it's an unconditional group difference, and
it differs from the regression coefficient on `sex` above because the regression holds experience,
education, and occupation fixed while the raw gap does not. Which one you want depends on the
question you're asking. That distinction — and how to get it right when you can't just "control for
everything" by hand — is where the rest of the semester lives.

---
## Quick reference

| Task | Tool |
|---|---|
| Row/column sums | `arr.sum(axis=0)` / `axis=1` |
| Avoid explicit loops | vectorized ops, broadcasting |
| Solve $\hat\beta$ without inverting | `np.linalg.lstsq(X, y, rcond=None)` |
| Reproducible randomness | `np.random.default_rng(seed)` |
| Label vs. position indexing | `.loc` (inclusive) vs `.iloc` (exclusive) |
| Row-wise custom function | `.apply` (slow) — prefer vectorized `.assign` |
| Missing data | `.isna()`, `.dropna()`, `.fillna()` |
| Per-group summary | `.groupby(...).agg(name=('col','func'))` |
| Per-row value within group | `.groupby(...)[col].transform(func)` |
| Join on a key | `pd.merge(..., on=, how=)` — check row count after! |
| Stack frames | `pd.concat([...], axis=0)` |
| Many-level categorical → dummies | `pd.get_dummies(..., sparse=True)` |
| Wide ↔ long | `.melt()` / `.pivot()` |
| Distribution functions | `stats.norm.{pdf,cdf,ppf,rvs}` |
| Two-sample test | `stats.ttest_ind` |
| OLS with full output | `sm.OLS(y, sm.add_constant(X)).fit()` |
| Robust SEs | `.fit(cov_type='HC1')` (or HC0/HC2/HC3) |
| R-style formulas | `statsmodels.formula.api.ols('y ~ x1*x2', data=df)` |

